In [1]:
# Imports
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/tmp/ipykernel_3528/2669226731.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'telco-customer-churn' dataset.
First 5 records:    customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic            

In [3]:
# Rebuild preprocessed data from raw CSV

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
df_encoded = pd.get_dummies(df.drop('customerID', axis=1), columns=cat_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

In [4]:
# =============================================================================
# TASK 1: Prepare and Scale the Data
# =============================================================================

# Step 2 — Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 3 — Scale with StandardScaler (fit on train, transform both)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Train shape: {X_train_s.shape}, Test shape: {X_test_s.shape}")

Train shape: (5634, 30), Test shape: (1409, 30)


In [5]:
# =============================================================================
# TASK 2: Train an SVM Classifier
# =============================================================================

# Step 1 — Train and time it
start = time.time()
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_s, y_train)
svm_time = time.time() - start

# Step 2 — Evaluate
y_pred_svm = svm.predict(X_test_s)
svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)

print(f"SVM Accuracy: {svm_accuracy:.4f}")
print(f"SVM F1: {svm_f1:.4f}")
print(f"SVM Training Time: {svm_time:.2f}s")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.7928
SVM F1: 0.5562
SVM Training Time: 2.66s
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1035
           1       0.64      0.49      0.56       374

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [6]:
# =============================================================================
# TASK 3: Train KNN with K=5
# =============================================================================

# Step 1 — Train and time it
start = time.time()
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_s, y_train)
knn5_time = time.time() - start

# Step 2 — Evaluate
y_pred_knn5 = knn5.predict(X_test_s)
knn5_accuracy = accuracy_score(y_test, y_pred_knn5)
knn5_f1 = f1_score(y_test, y_pred_knn5)

print(f"KNN (K=5) Accuracy: {knn5_accuracy:.4f}")
print(f"KNN (K=5) F1: {knn5_f1:.4f}")
print(f"KNN (K=5) Training Time: {knn5_time:.2f}s")
print(classification_report(y_test, y_pred_knn5))

KNN (K=5) Accuracy: 0.7473
KNN (K=5) F1: 0.5123
KNN (K=5) Training Time: 0.01s
              precision    recall  f1-score   support

           0       0.82      0.84      0.83      1035
           1       0.53      0.50      0.51       374

    accuracy                           0.75      1409
   macro avg       0.67      0.67      0.67      1409
weighted avg       0.74      0.75      0.75      1409



In [7]:
# =============================================================================
# TASK 4: Experiment with Different K Values
# =============================================================================

# Step 1 — Train KNN with K=3 and K=10
start = time.time()
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train_s, y_train)
knn3_time = time.time() - start
y_pred_knn3 = knn3.predict(X_test_s)
knn3_accuracy = accuracy_score(y_test, y_pred_knn3)
knn3_f1 = f1_score(y_test, y_pred_knn3)

start = time.time()
knn10 = KNeighborsClassifier(n_neighbors=10)
knn10.fit(X_train_s, y_train)
knn10_time = time.time() - start
y_pred_knn10 = knn10.predict(X_test_s)
knn10_accuracy = accuracy_score(y_test, y_pred_knn10)
knn10_f1 = f1_score(y_test, y_pred_knn10)

print(f"KNN (K=3)  Accuracy: {knn3_accuracy:.4f}  F1: {knn3_f1:.4f}")
print(f"KNN (K=5)  Accuracy: {knn5_accuracy:.4f}  F1: {knn5_f1:.4f}")
print(f"KNN (K=10) Accuracy: {knn10_accuracy:.4f}  F1: {knn10_f1:.4f}")

# Step 2 — Comparison table
k_comparison_df = pd.DataFrame({
    'K Value': [3, 5, 10],
    'Accuracy': [knn3_accuracy, knn5_accuracy, knn10_accuracy],
    'F1': [knn3_f1, knn5_f1, knn10_f1]
})
print(k_comparison_df.to_string(index=False))
k_comparison_df

# Step 3 — Identify best K programmatically (by F1)
best_k_row = k_comparison_df.loc[k_comparison_df['F1'].idxmax()]
print(f"Best K by F1: {int(best_k_row['K Value'])} "
      f"(Accuracy: {best_k_row['Accuracy']:.4f}, F1: {best_k_row['F1']:.4f})")

# Map back to the actual fitted model/predictions/time for Task 5.
knn_models = {3: (knn3, knn3_accuracy, knn3_f1, knn3_time),
              5: (knn5, knn5_accuracy, knn5_f1, knn5_time),
              10: (knn10, knn10_accuracy, knn10_f1, knn10_time)}
best_k = int(best_k_row['K Value'])
best_knn_model, best_knn_accuracy, best_knn_f1, best_knn_time = knn_models[best_k]

KNN (K=3)  Accuracy: 0.7438  F1: 0.5128
KNN (K=5)  Accuracy: 0.7473  F1: 0.5123
KNN (K=10) Accuracy: 0.7729  F1: 0.5266
 K Value  Accuracy       F1
       3  0.743790 0.512821
       5  0.747339 0.512329
      10  0.772889 0.526627
Best K by F1: 10 (Accuracy: 0.7729, F1: 0.5266)


In [8]:
# =============================================================================
# TASK 5: SVM vs Best KNN — Full Comparison
# =============================================================================

# Step 1 — Summary table
summary_df = pd.DataFrame({
    'Model': ['SVM (RBF)', f'KNN (K={best_k})'],
    'Accuracy': [svm_accuracy, best_knn_accuracy],
    'F1': [svm_f1, best_knn_f1],
    'Training Time (s)': [svm_time, best_knn_time]
})
print(summary_df.to_string(index=False))
summary_df

# Measure KNN prediction time to make Task 5's
# point concrete — useful evidence for your discussion paragraph.
start = time.time()
_ = svm.predict(X_test_s)
svm_predict_time = time.time() - start

start = time.time()
_ = best_knn_model.predict(X_test_s)
knn_predict_time = time.time() - start

print(f"SVM prediction time:  {svm_predict_time:.4f}s")
print(f"KNN prediction time:  {knn_predict_time:.4f}s")

     Model  Accuracy       F1  Training Time (s)
 SVM (RBF)  0.792761 0.556231           2.655733
KNN (K=10)  0.772889 0.526627           0.010231
SVM prediction time:  0.9679s
KNN prediction time:  0.1996s
